# Stage 7 -- 拟时序分析 (Pseudotime & Root Identification)

本 notebook 对上皮谱系做拟时序分析，包含五个分析模块：

1. **转录组熵 (Transcriptomic Entropy)** -- 每细胞表达熵作为分化潜能代理指标
2. **CytoTRACE** -- cellrank `CytoTRACEKernel`（纯 Python，不依赖 R）
3. **多指标 Root Cluster 识别** -- Z-score 综合多个指标（熵/CytoTRACE/干细胞 marker）定拟时序起点
4. **Root 可视化** -- UMAP 高亮 + 综合得分排序条形图
5. **Monocle3 轨迹推断** -- subprocess Rscript 桥接（R 包未安装或不可用时优雅跳过）

## 生物学背景

**为什么做拟时序分析？** 单细胞数据是快照，每个细胞的 mRNA 代表分化过程的某一瞬间。
拟时序分析将细胞沿着"分化轨迹"排序，重建从干细胞/祖细胞（root）到终末分化细胞的
动态过程。对于胃"炎-癌"转化场景，拟时序可以揭示正常上皮->化生->异型增生的
分子轨迹，识别驱动恶性转化的关键分支点。

**为什么上皮谱系？** 胃癌起源于上皮细胞（腺上皮->肠化->异型增生->癌）。
间质和免疫细胞有自己的独立分化轨迹，混合分析会模糊上皮恶变的信号。
在运行本 notebook 前，应先通过 stage6 完成细胞类型注释，
然后将 `EPITHELIAL_CLUSTERS` 设为目标上皮簇。
若无注释，默认对所有细胞运行（PI 后续可按谱系筛选）。

**转录组熵作为分化潜能代理**：高熵 = 表达模式"扁平"（许多基因低表达，
少有基因极高表达）-> 未分化/干细胞样状态。低熵 = 少数基因极高表达，
多数基因沉默 -> 终末分化状态。这是基于信息论的优雅指标，不依赖先验 marker，
已在多项研究中验证与干细胞潜能的相关性。

**CytoTRACE**：基于基因计数（每个细胞检测到的基因数）与共表达模式推断
分化潜能。核心假设是：未分化细胞转录组更"嘈杂"（更多基因被低水平检测到），
因此每个细胞的基因计数与分化程度呈负相关。cellrank 的 `CytoTRACEKernel`
是该算法的 Python 实现，结果与 R 版高度一致。

**多指标 Root 识别**：单一指标（如仅看 CytoTRACE）可能被噪音误导。
综合熵、CytoTRACE 和干细胞 marker 表达，对各聚类簇计算 Z-score
加权综合得分，得分最高的簇定为拟时序起点（root cluster）。
多指标交叉验证策略的稳健性显著优于单指标。

**Monocle3**：基于图学习的轨迹推断工具。从细胞表达数据构建 principal graph，
在图上识别叶节点和分支点，然后从用户指定的 root cells 出发计算 pseudotime。
Monocle3 擅长处理复杂的分叉轨迹，是目前最广泛使用的拟时序工具之一。

产出：`adata.obs`（`entropy` / `cytotrace_score` / `root_score` / `pseudotime_v1`）
+ figures 保存到 `results/figures/`。


In [ ]:
# === PARAMS ===
# UPSTREAM_PATH           -- stage6/6.5 注释结果 h5ad（含 CLUSTER_KEY 或 CELL_TYPE_COL）
# OUTPUT_PATH             -- 本 notebook 产出 checkpoint
# CLUSTER_KEY             -- 用于按簇聚合指标的 obs 列（root 识别时按此列汇总）
# CELL_TYPE_COL           -- 优先使用的细胞类型列（如 stage6 已完成注释）
# EPITHELIAL_CLUSTERS     -- 上皮谱系 cluster ID 列表；None=全体细胞
#                            重要提醒：PI 应根据实际的标记基因/细胞类型注释，
#                            确认哪些 cluster 属于上皮谱系后填写此参数。
# CYTO_N_NEIGHBORS        -- CytoTRACE 邻接图的邻居数
# WEIGHT_CYTOTRACE        -- Root 评分中 CytoTRACE 的权重
# WEIGHT_ENTROPY          -- Root 评分中转录组熵的权重
# WEIGHT_STEM_MARKER      -- Root 评分中干细胞 marker 的权重（如不可用则自动重分配）
# STEM_MARKERS            -- 已知干细胞/祖细胞 marker 基因列表
#                            基因需在 adata.var_names 中存在；不存在的自动跳过
# MONOCLE3_WORK_DIR       -- Monocle3 Rscript 临时工作目录
# MONOCLE3_NUM_DIM        -- Monocle3 PCA 降维维度
# MONOCLE3_CORES          -- Monocle3 并行线程数
# RSCRIPT_BIN             -- Rscript 可执行文件路径

UPSTREAM_PATH = "results/nancang_stage6_annotated_v1.h5ad"
OUTPUT_PATH   = "results/stage7_pseudotime.h5ad"

CLUSTER_KEY   = "leiden_res_0.6"
CELL_TYPE_COL = "cell_type_final_v1"

EPITHELIAL_CLUSTERS = None   # None = 全体细胞；PI 按需设如 ["0", "3", "5"]

CYTO_N_NEIGHBORS = 30

WEIGHT_CYTOTRACE   = 0.50   # CytoTRACE 对干性判别力最强
WEIGHT_ENTROPY     = 0.35   # 转录组熵：分化潜能代理指标
WEIGHT_STEM_MARKER = 0.15   # 干细胞 marker 平均表达

STEM_MARKERS = [
    "LGR5", "SOX9", "MKI67", "OLFM4", "TERT",
    "AXIN2", "LRIG1", "TFF2", "MUC6", "MUC5AC",
]
# 以上为胃上皮干细胞/祖细胞常用 marker，PI 应根据实际数据与生物学知识调整。
# 基因不存在时自动跳过，不影响其余指标的计算。

MONOCLE3_WORK_DIR = "results/_monocle3_tmp"
MONOCLE3_NUM_DIM  = 50
MONOCLE3_CORES    = 1
# RSCRIPT_BIN 通过 platform 模块统一解析（ADR-0010）
# rscript_bin() 找不到 Rscript 时会抛 RuntimeError，
# 用 try-except 优雅降级：设 RSCRIPT_BIN=None 让后续 Monocle3 步骤跳过。
from scrna_integration.platform import rscript_bin
try:
    RSCRIPT_BIN = rscript_bin()
except RuntimeError:
    RSCRIPT_BIN = None


In [ ]:
# 确保框架 src/ 在 sys.path 并切换到项目根目录。
# 多级回退策略：nbconvert/conda run 的 CWD 不稳定，
# 先试 CWD，再试从 notebooks/stage7/ 回退两级，最后用 notebook 自身路径推算。
import sys, os, gc
_root = os.getcwd()
_root_candidates = [
    _root,
    os.path.abspath(os.path.join(_root, "..")),
    os.path.abspath(os.path.join(_root, "..", "..")),
]

for _cand in _root_candidates:
    if os.path.isdir(os.path.join(_cand, "src", "scrna_integration")):
        _root = _cand
        break
else:
    _root = os.environ.get("PROJECT_ROOT", _root)

if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
os.makedirs(MONOCLE3_WORK_DIR, exist_ok=True)
print(f"PROJECT_ROOT: {_root}")
print(f"src 存在: {os.path.isdir(os.path.join(_root, 'src', 'scrna_integration'))}")


In [ ]:
# 导入依赖。
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cellrank as cr
import subprocess
import shutil
import warnings

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)

print(f"scanpy {sc.__version__}  |  cellrank {cr.__version__}  |  numpy {np.__version__}")


In [ ]:
# 加载上游 stage6 输出。
# 契约：需包含 CLUSTER_KEY 列 + embedding（X_umap 或 X_scVI/X_scANVI）。
# 细胞类型列 CELL_TYPE_COL 优先使用 -- 若有注释则按它筛选谱系与命名。
print(f"加载上游: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"obs 列: {list(adata.obs.columns)}")

# 确定实际使用的分组列
if CELL_TYPE_COL in adata.obs.columns:
    _group_col = CELL_TYPE_COL
    print(f"使用细胞类型列: {CELL_TYPE_COL}")
    _ct = adata.obs[CELL_TYPE_COL].dropna().astype(str)
    print(f"  细胞类型: {sorted(_ct.unique())}")
elif CLUSTER_KEY in adata.obs.columns:
    _group_col = CLUSTER_KEY
    print(f"CELL_TYPE_COL 不存在，fallback 到 CLUSTER_KEY: {CLUSTER_KEY}")
    print(f"  簇数: {adata.obs[CLUSTER_KEY].nunique()}")
else:
    raise KeyError(
        f"CELL_TYPE_COL '{CELL_TYPE_COL}' 和 "
        f"CLUSTER_KEY '{CLUSTER_KEY}' 都不在 obs 列中"
    )

# 检查 embedding
_has_umap = "X_umap" in adata.obsm
_has_scvi = any(
    k.startswith("X_scVI") or k.startswith("X_scANVI")
    for k in adata.obsm.keys()
)
print(f"X_umap: {_has_umap}  |  scVI/scANVI embedding: {_has_scvi}")
print(f"obsm keys: {list(adata.obsm.keys())}")

# 上皮谱系筛选 -- 由 PI 通过 EPITHELIAL_CLUSTERS 指定
if EPITHELIAL_CLUSTERS is not None:
    _epi_mask = adata.obs[_group_col].astype(str).isin(
        [str(c) for c in EPITHELIAL_CLUSTERS]
    )
    _n_before = adata.n_obs
    if not _epi_mask.any():
        raise ValueError(
            f"EPITHELIAL_CLUSTERS={EPITHELIAL_CLUSTERS} 不匹配任何细胞。"
            f"可用的 {_group_col} 值: "
            f"{sorted(adata.obs[_group_col].dropna().astype(str).unique())}"
        )
    adata = adata[_epi_mask].copy()
    print(
        f"上皮谱系筛选: {_n_before:,} -> {adata.n_obs:,} 细胞 "
        f"(保留 {_group_col}: {EPITHELIAL_CLUSTERS})"
    )
else:
    print(
        "EPITHELIAL_CLUSTERS=None，对所有细胞做拟时序。"
        "PI 可在完成细胞类型注释后指定上皮 cluster 重新分析。"
    )


## 1. 转录组熵 -- 分化潜能的信息论代理

**为什么算熵？** Shannon 熵衡量细胞表达模式的"集中度"：
- **高熵**：转录本均匀分布在许多基因上 -> 未分化/多能状态（大量"低水平备用转录"）
- **低熵**：转录本集中在少数基因上 -> 终末分化状态（特异功能的基因极高表达）

相比传统方法（如找已知干性 marker 基因的表达均值），熵是无监督指标 --
不需要任何先验的 marker 基因列表，仅从表达数据本身判断分化程度。
这在真实临床样本中特别有价值：不同患者的"干细胞"未必表达同一套 marker。

**实现**：纯 numpy，无 R 依赖。逻辑来自 student-code `4.3_*.py`
的 `compute_transcriptome_entropy_from_matrix`，按本项目规范重写。

算法：对每个细胞 i，计算表达概率 p_i = X_i / sum(X_i)，然后
entropy_i = -sum(p_i * log2(p_i + 1e-12))。
对稀疏矩阵做 CSR 优化，避免 dense 化导致的内存爆炸。


In [ ]:
# 转录组熵 -- 逐细胞计算 Shannon 熵作为分化潜能的代理指标。
# 对稀疏矩阵走 CSR 乘法路径（避免 toarray() 导致 OOM），
# 对 dense 矩阵走 numpy 向量化路径。
# 为什么用 counts 层而非 log-normalized？熵的定义要求原始计数或频率分布；
# log-normalized 数据中负值会破坏概率语义（P_i = count_i / sum(counts)）。
# 优先使用 adata.layers["counts"]（原始计数），其次 adata.raw.X，最后 adata.X。

def _get_count_matrix(adata):
    """获取计数矩阵 -- 优先 counts layer，其次 raw，最后 X。"""
    if "counts" in adata.layers:
        return adata.layers["counts"]
    if adata.raw is not None:
        return adata.raw.X
    return adata.X

def _entropy_from_sparse(X_csr):
    """从稀疏 CSR 矩阵计算每行（每细胞）的 Shannon 熵。
    走 CSR 逐行乘法路径，避免对整个矩阵做 dense 化。
    复杂度 O(nnz)，对大矩阵内存友好。"""
    # 逐行归一化：P = X / row_sum（保持稀疏）
    row_sums = np.asarray(X_csr.sum(axis=1)).ravel()
    row_sums[row_sums == 0] = 1.0  # 全零细胞避免除零 -- 熵定义为 0
    # CSR 逐元素：P * log2(P + 1e-12)，累加得 -sum(P * log2(P))
    P = X_csr.copy()
    P.data = P.data / row_sums[P.nonzero()[0]]  # 每行归一化
    P.data = P.data * np.log2(P.data + 1e-12)    # P * log2(P)
    entropy = -np.asarray(P.sum(axis=1)).ravel()
    return entropy

def _entropy_from_dense(X):
    """从 dense 矩阵计算熵 -- 向量化路径，适合小数据集。"""
    X = np.asarray(X, dtype=np.float64)
    row_sums = X.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    P = X / row_sums
    return -np.sum(P * np.log2(P + 1e-12), axis=1)

_Xc = _get_count_matrix(adata)
if sp.issparse(_Xc):
    _Xc_csr = _Xc.tocsr()
    adata.obs["entropy"] = _entropy_from_sparse(_Xc_csr)
    adata.obs["n_genes_by_counts"] = np.asarray((_Xc_csr > 0).sum(axis=1)).ravel()
else:
    adata.obs["entropy"] = _entropy_from_dense(_Xc)
    adata.obs["n_genes_by_counts"] = (np.asarray(_Xc) > 0).sum(axis=1)

print("转录组熵已写入 adata.obs['entropy']")
print(f"  均值: {adata.obs['entropy'].mean():.3f}")
print(f"  范围: [{adata.obs['entropy'].min():.3f}, {adata.obs['entropy'].max():.3f}]")
print(f"  n_genes_by_counts 均值: {adata.obs['n_genes_by_counts'].mean():.0f}")

# 解释：在胃上皮谱系中，预期干细胞/祖细胞（如 LGR5+ 细胞）有更高的熵值，
# 而终末分化的腺细胞（如壁细胞、主细胞、杯状细胞）有更低的熵值。
# 但熵值也受技术因素影响（测序深度、细胞质量），
# 因此不单独用于 root 判断，而是作为多指标综合评分的一个维度。


## 2. CytoTRACE -- 分化潜能评分

**为什么用 CytoTRACE？** 它在多个独立基准测试中与 gold-standard 的干性排序
高度一致（Spearman rho > 0.9），且不依赖 marker 基因的先验知识。
其核心算法分两步：
1. 以每细胞的基因计数（n_genes_by_counts）为初始分化潜能估计
2. 通过局部邻域平滑（"传递"基因计数信号到转录相似的邻居）修正技术噪音

cellrank 的 `CytoTRACEKernel` 是对 R/CytoTRACE 的纯 Python 复现，
本项目已验证其可用性。

**为什么需要先建 neighbors？** CytoTRACE 的平滑步骤依赖细胞的 KNN 邻接图。
如果上游 stage4/stage5 已经建过 neighbors（`adata.uns["neighbors"]` 存在），
则直接复用，避免重复计算。否则用 scVI/scANVI 潜空间或 PCA 重建。


In [ ]:
# CytoTRACE -- cellrank CytoTRACEKernel，纯 Python 不依赖 R。
# 确保有 neighbors（CytoTRACE 需要细胞的局部邻域来平滑基因计数信号）。
# 优先复用上游已建的 neighbors，避免重复计算。

if "neighbors" not in adata.uns:
    print("未检测到 neighbors -- 正在构建...")
    # 选择 embedding 建 neighbors：scANVI > scVI > PCA
    _use_rep = None
    for _key in adata.obsm.keys():
        if _key.lower().startswith("x_scanvi"):
            _use_rep = _key
            break
    if _use_rep is None:
        for _key in adata.obsm.keys():
            if _key.lower().startswith("x_scvi"):
                _use_rep = _key
                break
    if _use_rep is not None:
        print(f"  使用 embedding: {_use_rep}")
        sc.pp.neighbors(adata, use_rep=_use_rep, n_neighbors=CYTO_N_NEIGHBORS)
    elif "X_pca" in adata.obsm:
        print("  使用 X_pca 建 neighbors")
        sc.pp.neighbors(adata, n_neighbors=CYTO_N_NEIGHBORS)
    else:
        print("  无 embedding，先跑 PCA...")
        sc.tl.pca(adata, n_comps=50)
        sc.pp.neighbors(adata, n_neighbors=CYTO_N_NEIGHBORS)
else:
    print("复用上游已建的 neighbors")

# 运行 CytoTRACE。显式指定 layer="X" 避免 cellrank 默认找 layer["imputed"]。
ctk = cr.kernels.CytoTRACEKernel(adata)
ctk.compute_cytotrace(layer="X")

# CytoTRACEKernel 可能将结果写入不同字段名，统一收束到 cycotrace_score
if hasattr(ctk, "cytotrace"):
    adata.obs["cytotrace_score"] = np.asarray(ctk.cytotrace).ravel().astype(np.float32)
elif "ct_score" in adata.obs.columns:
    adata.obs["cytotrace_score"] = adata.obs["ct_score"].astype(np.float32)
else:
    # 搜索可能的列名
    _candidates = [c for c in adata.obs.columns if "cyto" in c.lower() or "trace" in c.lower()]
    if _candidates:
        adata.obs["cytotrace_score"] = pd.to_numeric(
            adata.obs[_candidates[0]], errors="coerce"
        ).astype(np.float32)
    else:
        raise RuntimeError("CytoTRACE 运行后未找到分数字段，请检查 cellrank 版本。")

print("CytoTRACE 完成，已写入 adata.obs['cytotrace_score']")
print(f"  均值: {adata.obs['cytotrace_score'].mean():.4f}")
print(f"  范围: [{adata.obs['cytotrace_score'].min():.4f}, "
      f"{adata.obs['cytotrace_score'].max():.4f}]")
# 解释：CytoTRACE score 高 -> 更接近干细胞/祖细胞（分化潜能高）。
# 对于胃上皮谱系，预期 LGR5+ 干细胞所在簇的 score 最高，
# 终末分化细胞（如壁细胞分泌酸的专门化细胞）score 最低。


## 3. 多指标 Root Cluster 识别

**为什么需要识别 root cluster？** 拟时序分析的起点（root）直接决定
pseudotime 的生物学解释力。理想起点是"最接近干细胞/祖细胞状态"的细胞群。
单一指标（如仅看 CytoTRACE）可能被技术噪音或组织特异表达模式误导。

**多指标交叉验证策略**（逻辑重写自 student-code `4.4_*.py`）：
1. **按簇聚合** 各簇的熵、CytoTRACE、干细胞 marker 的中位值
2. **Z-score 标准化** 每个指标转换为 z-score（消除量纲差异）
3. **加权综合** 得分 = sum(w_i * z_i)，权重由 PARAMS 指定
4. 得分最高的簇定为 `root_cluster`

**指标的生物学方向**：
- CytoTRACE：越高越接近干细胞（正向）
- 转录组熵：越高越接近未分化状态（正向）
- 干细胞 marker 平均表达：在干/祖细胞中高表达（正向）

**自适应权重**：如果指定的干细胞 marker 基因在数据集中不存在，
则将其权重按比例重新分配给 CytoTRACE 和熵，确保总权重始终为 1.0。


In [ ]:
# 多指标 Root Cluster 识别 -- Z-score 综合得分，确定拟时序起点。
# 逻辑重写自 student-code workflow_for_pseudotime/4.4_*.py：
# summarize_clusters() + score() + pick_root_cell()，
# 完全按本项目规范重写（去硬编码路径 + 自适应权重 + 中文注释讲 why）。

# Step 1: 检查干细胞 marker 基因在数据集中的存在性
_present_markers = [g for g in STEM_MARKERS if g in adata.var_names]
_missing_markers = [g for g in STEM_MARKERS if g not in adata.var_names]
print(f"干细胞 marker: {len(_present_markers)}/{len(STEM_MARKERS)} 个基因存在于数据集中")
if _missing_markers:
    print(f"  缺失基因: {_missing_markers}")

# Step 2: 自适应权重 -- 如果 marker 全部缺失，权重重分配
if len(_present_markers) == 0:
    _w_ct = WEIGHT_CYTOTRACE + WEIGHT_STEM_MARKER * 0.6
    _w_en = WEIGHT_ENTROPY + WEIGHT_STEM_MARKER * 0.4
    _w_mk = 0.0
    print(f"干细胞 marker 全部缺失，权重重分配: CytoTRACE={_w_ct:.2f}, Entropy={_w_en:.2f}")
else:
    _w_ct = WEIGHT_CYTOTRACE
    _w_en = WEIGHT_ENTROPY
    _w_mk = WEIGHT_STEM_MARKER
    # 计算每细胞的干细胞 marker 均值（对 count 矩阵）
    if "counts" in adata.layers:
        _stem_expr = np.asarray(
            adata[:, _present_markers].layers["counts"].mean(axis=1)
        ).ravel()
    elif adata.raw is not None:
        _stem_expr = np.asarray(
            adata.raw[:, _present_markers].X.mean(axis=1)
        ).ravel()
    else:
        _stem_expr = np.asarray(
            adata[:, _present_markers].X.mean(axis=1)
        ).ravel()
    adata.obs["stem_marker_mean"] = _stem_expr

# Step 3: Z-score 标准化函数
def _zscore(series):
    """将 series 转为 z-score。"""
    x = pd.to_numeric(series, errors="coerce").astype(float)
    mu = np.nanmean(x)
    sd = np.nanstd(x)
    if sd == 0 or np.isnan(sd):
        return pd.Series(np.zeros(len(x)), index=series.index)
    z = (x - mu) / sd
    return z

# Step 4: 按簇聚合各指标的中位值
_gcol = _group_col  # 继承自上游加载 cell
_groups = adata.obs[_gcol].astype(str)
# NaN 提示：.astype(str) 会把 NaN 变成字面 "nan" 成为有效簇
if adata.obs[_gcol].isna().any():
    _n_nan = adata.obs[_gcol].isna().sum()
    print(f"WARNING: {_gcol} 中有 {_n_nan} 个 NaN 细胞（未注释），"
          f"转为 'nan' 后 root 识别可能被误导。"
          f"建议设 EPITHELIAL_CLUSTERS 排除或先 dropna。")
_cluster_ids = sorted(_groups.unique())

_records = []
for _clu in _cluster_ids:
    _mask = _groups == _clu
    _rec = {"cluster": _clu, "n_cells": _mask.sum()}

    if "cytotrace_score" in adata.obs.columns:
        _rec["cytotrace_median"] = adata.obs.loc[_mask, "cytotrace_score"].median()
    else:
        _rec["cytotrace_median"] = np.nan

    if "entropy" in adata.obs.columns:
        _rec["entropy_median"] = adata.obs.loc[_mask, "entropy"].median()
    else:
        _rec["entropy_median"] = np.nan

    if "stem_marker_mean" in adata.obs.columns:
        _rec["stem_marker_median"] = adata.obs.loc[_mask, "stem_marker_mean"].median()
    else:
        _rec["stem_marker_median"] = np.nan

    _records.append(_rec)

_df_clusters = pd.DataFrame(_records)

# Step 5: Z-score 加权综合得分
_df_clusters["z_cytotrace"] = _zscore(_df_clusters["cytotrace_median"])
_df_clusters["z_entropy"]   = _zscore(_df_clusters["entropy_median"])
if "stem_marker_median" in _df_clusters.columns and _w_mk > 0:
    _df_clusters["z_stem_marker"] = _zscore(_df_clusters["stem_marker_median"])
    _df_clusters["root_score"] = (
        _w_ct * _df_clusters["z_cytotrace"]
        + _w_en * _df_clusters["z_entropy"]
        + _w_mk * _df_clusters["z_stem_marker"]
    )
else:
    _df_clusters["root_score"] = (
        _w_ct * _df_clusters["z_cytotrace"]
        + _w_en * _df_clusters["z_entropy"]
    )

_df_clusters = _df_clusters.sort_values("root_score", ascending=False)
_df_clusters["rank"] = range(1, len(_df_clusters) + 1)

# Step 6: 确定 top root cluster 并映射回每个细胞
_top_cluster = _df_clusters.iloc[0]["cluster"]
_root_score_map = dict(zip(_df_clusters["cluster"], _df_clusters["root_score"]))
adata.obs["root_score"] = _groups.map(_root_score_map).astype(np.float32)
adata.uns["root_cluster"] = str(_top_cluster)
# 同时写入 obs 列供 Monocle3 R 脚本通过 cell_meta CSV 读取
adata.obs["root_cluster"] = _groups.astype(str)

_rs_min = adata.obs["root_score"].min()
_rs_max = adata.obs["root_score"].max()
print(f"Root 识别完成: top cluster = {_top_cluster}")
print(f"  root_score 范围: [{_rs_min:.3f}, {_rs_max:.3f}]")
print(f"\n各簇综合得分（前 5）:")
print(_df_clusters[["cluster", "n_cells", "root_score", "rank"]].head(5).to_string(index=False))

# Step 7: 在 top cluster 内选择代表性 root cell
# 为什么选一个代表细胞？Monocle3 的 order_cells() 接受 root_cells 参数；
# scanpy 的 dpt 也需要 iroot 索引。在 top cluster 中选熵/CytoTRACE 综合
# 得分最高的单个细胞作为 root cell，确保轨迹推断从"最干性"的细胞出发。
_top_mask = _groups == _top_cluster
_top_obs = adata.obs.loc[_top_mask]
# 在 top cluster 内用 min-max 归一化后加权选最佳细胞
if "cytotrace_score" in _top_obs.columns and "entropy" in _top_obs.columns:
    _ct_mm = (_top_obs["cytotrace_score"] - _top_obs["cytotrace_score"].min()) / (
        _top_obs["cytotrace_score"].max() - _top_obs["cytotrace_score"].min() + 1e-12
    )
    _en_mm = (_top_obs["entropy"] - _top_obs["entropy"].min()) / (
        _top_obs["entropy"].max() - _top_obs["entropy"].min() + 1e-12
    )
    _cell_score = 0.6 * _ct_mm + 0.4 * _en_mm
    _root_cell = _cell_score.idxmax()
elif "cytotrace_score" in _top_obs.columns:
    _root_cell = _top_obs["cytotrace_score"].idxmax()
elif "entropy" in _top_obs.columns:
    _root_cell = _top_obs["entropy"].idxmax()
else:
    _root_cell = _top_obs.index[0]

adata.uns["root_cell"] = str(_root_cell)
if _root_cell in adata.obs_names:
    adata.uns["iroot"] = int(np.where(adata.obs_names == _root_cell)[0][0])
    print(f"root cell: {_root_cell}  (iroot={adata.uns['iroot']})")
else:
    print(f"WARNING: root cell {_root_cell} 不在 adata.obs_names 中")

# 保存簇级得分表
_root_csv = "results/tables/stage7_root_cluster_scores.csv"
_df_clusters.to_csv(_root_csv, index=False)
print(f"簇级得分表已保存: {_root_csv}")


## 4. Root 可视化

**为什么可视化 root 结果？** 让 PI 直观判断 root cluster 选择是否合理。
可视化从两个角度交叉验证：
1. **综合得分条形图**：展示各簇的 root_score 排名——一眼看出哪个簇显著领先
2. **UMAP 高亮**：在现有 UMAP 上标出 top root cluster（蓝色区域）+ root cell（红色星号）
   如果 top cluster 位于 UMAP 图谱的"源头"位置（分化轨迹的起点），
   说明多指标识别结果与视觉直觉一致。

重写自 student-code `4.5_*.py`，按本项目规范简化为 notebook 内嵌可视化。


In [ ]:
# Root 可视化：综合得分条形图 + UMAP 高亮。
# 重写自 student-code workflow_for_pseudotime/4.5_visualize_root_cluster.py，
# 去 CLI 包装、去硬编码路径、注释中文化。

_top_cluster = adata.uns.get("root_cluster", None)
_root_cell = adata.uns.get("root_cell", None)
# 复用上游的 root 得分表
_root_csv = "results/tables/stage7_root_cluster_scores.csv"
if os.path.exists(_root_csv):
    _df_score = pd.read_csv(_root_csv)
    _df_score["cluster"] = _df_score["cluster"].astype(str)
else:
    _df_score = None

# Fig 1: 综合得分条形图
if _df_score is not None:
    fig, ax = plt.subplots(figsize=(10, 4))
    _df_sorted = _df_score.sort_values("root_score", ascending=False)
    _colors = ["#d62728" if str(c) == str(_top_cluster) else "#bdbdbd"
               for c in _df_sorted["cluster"]]
    ax.bar(range(len(_df_sorted)), _df_sorted["root_score"], color=_colors)
    ax.set_xticks(range(len(_df_sorted)))
    ax.set_xticklabels(_df_sorted["cluster"], rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("Root Score (Z-score weighted)")
    ax.set_xlabel("Cluster")
    ax.set_title(f"Root Cluster Score Ranking  |  Top: {_top_cluster}")
    ax.axhline(y=0, color="gray", lw=0.8, ls="--")
    # 标注 top cluster
    _top_idx = _df_sorted["cluster"].tolist().index(str(_top_cluster)) if str(_top_cluster) in _df_sorted["cluster"].tolist() else 0
    ax.annotate(
        f"Root: {_top_cluster}",
        xy=(_top_idx, _df_sorted.iloc[_top_idx]["root_score"]),
        xytext=(_top_idx + 1, _df_sorted.iloc[_top_idx]["root_score"] + 0.3),
        arrowprops=dict(arrowstyle="->", color="#d62728"),
        fontsize=10, color="#d62728", fontweight="bold",
    )
    plt.tight_layout()
    _bar_path = "results/figures/stage7_root_score_bar.png"
    fig.savefig(_bar_path, dpi=200, bbox_inches="tight")
    plt.close("all")
    print(f"Root 得分条形图已保存: {_bar_path}")
else:
    print("无 root 得分表，跳过条形图")

# Fig 2: UMAP 高亮 top root cluster + root cell
if _top_cluster is not None and "X_umap" in adata.obsm:
    fig, ax = plt.subplots(figsize=(7, 7))
    _coords = adata.obsm["X_umap"]

    # 背景细胞（浅灰）
    ax.scatter(_coords[:, 0], _coords[:, 1],
               c="lightgrey", s=3, alpha=0.4, rasterized=True, label="Other cells")

    # 高亮 top root cluster（蓝色）
    _top_mask = (_groups.astype(str) == str(_top_cluster)).values
    ax.scatter(_coords[_top_mask, 0], _coords[_top_mask, 1],
               c="#1f77b4", s=8, alpha=0.8, label=f"Root Cluster: {_top_cluster}")

    # 红色星号高亮 root cell
    if _root_cell is not None and _root_cell in adata.obs_names:
        _rc_idx = np.where(adata.obs_names == _root_cell)[0][0]
        ax.scatter(_coords[_rc_idx, 0], _coords[_rc_idx, 1],
                   c="red", marker="*", s=250, edgecolors="black",
                   linewidths=0.8, zorder=10, label="Root Cell")

    ax.set_title("Root Identification — UMAP")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.legend(loc="upper right", frameon=False, markerscale=2)
    plt.tight_layout()
    _umap_path = "results/figures/stage7_root_umap_highlight.png"
    fig.savefig(_umap_path, dpi=200, bbox_inches="tight")
    plt.close("all")
    print(f"Root UMAP 高亮图已保存: {_umap_path}")
elif "X_umap" not in adata.obsm:
    print("X_umap 不存在，跳过 UMAP 可视化。请确保上游 stage4 已跑 UMAP。")
else:
    print("root_cluster 未确定，跳过 UMAP 可视化")

# Fig 3: CytoTRACE vs Entropy 散点图（按 root_score 着色）
if "cytotrace_score" in adata.obs.columns and "entropy" in adata.obs.columns:
    fig, ax = plt.subplots(figsize=(7, 6))
    _sc = ax.scatter(
        adata.obs["cytotrace_score"],
        adata.obs["entropy"],
        c=adata.obs["root_score"],
        cmap="viridis", s=2, alpha=0.6, rasterized=True,
    )
    plt.colorbar(_sc, ax=ax, label="Root Score")
    ax.set_xlabel("CytoTRACE Score")
    ax.set_ylabel("Transcriptomic Entropy")
    ax.set_title("CytoTRACE vs Entropy — colored by Root Score")
    # 标注 top cluster 的位置
    if _top_cluster is not None:
        _top_vals = adata.obs.loc[_top_mask, ["cytotrace_score", "entropy"]]
        ax.scatter(
            _top_vals["cytotrace_score"], _top_vals["entropy"],
            s=20, facecolors="none", edgecolors="red", linewidths=0.8,
            label=f"Root Cluster: {_top_cluster}"
        )
        ax.legend(loc="upper right", frameon=False, markerscale=1.5)
    plt.tight_layout()
    _scatter_path = "results/figures/stage7_cytotrace_vs_entropy.png"
    fig.savefig(_scatter_path, dpi=200, bbox_inches="tight")
    plt.close("all")
    print(f"CytoTRACE vs Entropy 散点图已保存: {_scatter_path}")


## 5. Monocle3 轨迹推断 (R subprocess)

**为什么用 Monocle3？** 相比纯 Python 的 diffusion pseudotime (DPT)，Monocle3
的优势在于：自动学习轨迹的拓扑结构（树 vs 线）、识别分支点和叶节点、
支持从指定的 root cells 出发计算 pseudotime。对于胃上皮谱系可能存在
多分支分化（如主细胞 vs 壁细胞 vs 杯状细胞 lineage），Monocle3 更合适。

**为什么走 subprocess Rscript 而非 rpy2？** ADR-0007 明确：Monocle3 是
heavy R 工具，rpy2 + anndata2ri 在大对象转换上脆弱且依赖升级易崩。
subprocess 模式更鲁棒：写 .mtx/.csv -> `Rscript --vanilla` -> 读结果。
R 脚本是独立可调试的，不影响 Python 侧环境。（逻辑重写自 student-code
`4.3_*.py` 的 `write_monocle3_r_script` + `export_for_monocle3` + `run_monocle3`）

**R 守卫策略**：先检查 Rscript 可用性和 monocle3 R 包是否安装。
若不可用，优雅跳过并给出清晰提示（参考 stage2 SoupX 守卫模式）。
绝不硬装 R 包，绝不崩 notebook。

**数据导出**：Monocle3 需要原始计数矩阵（genes x cells 的 .mtz 文件）、
细胞元数据（含聚类标签和 sample_id）和基因注释。同时也导出已有的 UMAP
embedding，让 Monocle3 直接复用而非从头计算——既加速又保持与 Python 侧
UMAP 的可比性。


In [ ]:
# Monocle3 R 环境守卫：检查 Rscript 和 monocle3 R 包是否可用。
# 模式参考 stage2 SoupX 守卫——R 包不可用时优雅跳过，不崩 notebook。
# RSCRIPT_BIN 来自 PARAMS cell，已通过 try-except rscript_bin() 赋值；
# None 表示 Rscript 不可用，非 None 表示可用路径。
_monocle3_available = False
_r_available = RSCRIPT_BIN is not None
print(f"Rscript 可用: {_r_available}  (RSCRIPT_BIN={RSCRIPT_BIN})")

if _r_available:
    _check_cmd = [
        RSCRIPT_BIN, "--vanilla", "-e",
        "suppressPackageStartupMessages(library(monocle3)); cat('OK')",
    ]
    try:
        _res = subprocess.run(
            _check_cmd, capture_output=True, text=True, timeout=60,
        )
        if _res.returncode == 0 and "OK" in _res.stdout:
            _monocle3_available = True
            print("monocle3 R 包可用 -- Monocle3 轨迹推断将正常执行")
        else:
            print("monocle3 R 包不可用（load 失败或未安装）")
            print(f"  R stderr: {_res.stderr.strip()[:200]}")
    except (FileNotFoundError, subprocess.TimeoutExpired) as _e:
        print(f"Rscript 调用失败: {_e}")
else:
    print("Rscript 不可用，Monocle3 将跳过")

if not _monocle3_available:
    _monocle3_skip_msg = (
        "=" * 60 + "\n"
        "Monocle3 环境不可用，以下 cell 将优雅跳过。\n\n"
        "要启用 Monocle3 轨迹推断，请:\n"
        "  1. 确认 R 已安装（conda install -c conda-forge r-base）\n"
        "  2. 安装 monocle3 R 包:\n"
        "     R -e 'BiocManager::install(\"monocle3\")'\n"
        "  3. 重跑本 notebook\n"
        "=" * 60
    )
    print(_monocle3_skip_msg)


In [ ]:
# 导出 Monocle3 输入数据（.mtz + .csv）。
# Monocle3 需要原始计数矩阵（genes x cells 的 .mtz 格式）。
# 也导出 UMAP 坐标，让 Monocle3 直接复用 Python 侧的 embedding。

if _monocle3_available:
    shutil.rmtree(MONOCLE3_WORK_DIR, ignore_errors=True)
    os.makedirs(MONOCLE3_WORK_DIR, exist_ok=True)

    # 获取计数矩阵（优先 counts layer）
    if "counts" in adata.layers:
        _X_export = adata.layers["counts"]
    elif adata.raw is not None:
        _X_export = adata.raw[:, adata.var_names].X
    else:
        _X_export = adata.X

    # 写 .mtz（genes x cells -- Monocle3 需要基因在行）
    from scipy.io import mmwrite
    _X_gc = sp.csr_matrix(_X_export).T.tocoo()
    _mtx_path = os.path.join(MONOCLE3_WORK_DIR, "counts.mtx")
    mmwrite(_mtx_path, _X_gc)
    print(f"计数矩阵已导出: {_mtx_path}  (shape genes x cells: {_X_gc.shape})")

    # 细胞元数据
    _meta = adata.obs.copy()
    _meta["cell_id"] = adata.obs_names.astype(str)
    for _col in _meta.columns:
        if hasattr(_meta[_col], "cat"):
            _meta[_col] = _meta[_col].astype(str)
    _meta_path = os.path.join(MONOCLE3_WORK_DIR, "cell_meta.csv")
    _meta.to_csv(_meta_path, index=False)
    print(f"细胞元数据已导出: {_meta_path}  ({_meta.shape[0]} 细胞 x {_meta.shape[1]} 列)")

    # 基因注释
    _gene_df = pd.DataFrame({
        "gene_id": adata.var_names.astype(str),
        "gene_short_name": adata.var_names.astype(str),
    })
    _gene_path = os.path.join(MONOCLE3_WORK_DIR, "gene_anno.csv")
    _gene_df.to_csv(_gene_path, index=False)
    print(f"基因注释已导出: {_gene_path}  ({len(_gene_df)} 基因)")

    # UMAP 坐标（让 Monocle3 复用 Python 侧的 embedding）
    if "X_umap" in adata.obsm:
        _umap = pd.DataFrame(
            adata.obsm["X_umap"][:, :2],
            index=adata.obs_names.astype(str),
            columns=["UMAP1", "UMAP2"],
        )
        _umap.index.name = "cell_id"
        _umap.reset_index().to_csv(
            os.path.join(MONOCLE3_WORK_DIR, "existing_umap.csv"), index=False
        )
        print(f"UMAP 坐标已导出")
else:
    print("Monocle3 不可用，跳过数据导出")


In [ ]:
# Monocle3 R 脚本——内联生成然后 subprocess 调用。
# 逻辑重写自 student-code 4.3 和 11.2 的 R 脚本，按本项目规范简化：
# 去掉 hard-coded R_LIBS_USER、去掉 graph_test 和 find_gene_modules
# （纯轨迹推断，不做差异基因），复用 Python 侧的 UMAP embedding。

if _monocle3_available:
    _r_script = os.path.join(MONOCLE3_WORK_DIR, "run_monocle3.R")
    _r_code = '''#!/usr/bin/env Rscript
suppressPackageStartupMessages({
  library(monocle3)
  library(Matrix)
  library(data.table)
  library(igraph)
})

args <- commandArgs(trailingOnly = TRUE)
input_mtx    <- args[1]
input_meta   <- args[2]
input_gene   <- args[3]
input_umap   <- args[4]
output_prefix<- args[5]
num_dim      <- as.integer(args[6])
ncores       <- as.integer(args[7])

# 读入数据
expr <- readMM(input_mtx)
cell_meta <- fread(input_meta, data.table = FALSE)
rownames(cell_meta) <- cell_meta$cell_id
gene_anno <- fread(input_gene, data.table = FALSE)
rownames(gene_anno) <- gene_anno$gene_id

cds <- new_cell_data_set(
  expression_data = expr,
  cell_metadata = cell_meta,
  gene_metadata = gene_anno
)

# PCA 预降维
n_dim <- min(as.integer(num_dim), nrow(cds) - 1L, ncol(cds) - 1L)
n_dim <- max(2L, n_dim)
cds <- estimate_size_factors(cds)
cds <- preprocess_cds(cds, method = "PCA", num_dim = n_dim)

# 注入已有 UMAP，让 Monocle3 直接复用 Python 侧 embedding
if (file.exists(input_umap)) {
  umap_df <- fread(input_umap, data.table = FALSE)
  rownames(umap_df) <- umap_df$cell_id
  umap_mat <- as.matrix(umap_df[colnames(cds), c("UMAP1", "UMAP2")])
  rownames(umap_mat) <- colnames(cds)
  reducedDims(cds)$UMAP <- umap_mat
} else {
  cds <- reduce_dimension(cds, reduction_method = "UMAP",
                          preprocess_method = "PCA",
                          umap.fast_sgd = TRUE, cores = ncores)
}

# 聚类 + 图学习
cds <- cluster_cells(cds, reduction_method = "UMAP")
cds <- learn_graph(cds, use_partition = TRUE, close_loop = FALSE)

# 从 root cluster 选择 root cells —— 用 adata.obs 中标记的 root_cluster
cell_meta <- colData(cds)
if ("root_cluster" %in% colnames(cell_meta)) {
  top_cluster <- args[8]
  root_cells <- rownames(cell_meta)[as.character(cell_meta[["root_cluster"]]) == top_cluster]
} else {
  # fallback: 选 pseudotime 方向最接近起点的细胞
  g <- principal_graph(cds)[["UMAP"]]
  root_cells <- NULL
}

if (length(root_cells) > 0) {
  cds <- order_cells(cds, root_cells = root_cells)
} else {
  cds <- order_cells(cds)
}

# 提取叶节点与分支点
g <- principal_graph(cds)[["UMAP"]]
all_vertices <- igraph::V(g)$name
leaf_nodes  <- all_vertices[igraph::degree(g) == 1]
branch_nodes <- all_vertices[igraph::degree(g) > 2]

# 细胞-to-vertex 映射
closest_raw <- principal_graph_aux(cds)[["UMAP"]]$pr_graph_cell_proj_closest_vertex
if (is.matrix(closest_raw) || is.data.frame(closest_raw)) {
  if (!is.null(rownames(closest_raw)) && all(colnames(cds) %in% rownames(closest_raw))) {
    closest_vertex <- as.character(closest_raw[colnames(cds), 1])
  } else if (nrow(closest_raw) == ncol(cds)) {
    closest_vertex <- as.character(closest_raw[, 1])
  } else {
    closest_vertex <- as.character(closest_raw[1, ])
  }
} else {
  closest_vertex <- as.character(closest_raw)
}
names(closest_vertex) <- colnames(cds)

# 数字 vertex 名 -> Y_<index> 映射
if (!all(closest_vertex[!is.na(closest_vertex)] %in% all_vertices)) {
  if (all(grepl("^[0-9]+$", closest_vertex[!is.na(closest_vertex)]))) {
    closest_vertex <- paste0("Y_", closest_vertex)
  }
}

# 输出结果
pseudotime_vec <- pseudotime(cds)
cluster_vec <- tryCatch(as.character(clusters(cds)),
                        error = function(e) rep(NA_character_, ncol(cds)))
partition_vec <- tryCatch(as.character(partitions(cds)),
                          error = function(e) rep(NA_character_, ncol(cds)))

cell_out <- data.frame(
  cell_id = colnames(cds),
  pseudotime = as.numeric(pseudotime_vec),
  monocle3_cluster = cluster_vec,
  monocle3_partition = partition_vec,
  monocle3_closest_vertex = closest_vertex[colnames(cds)],
  is_leaf_direct = closest_vertex[colnames(cds)] %in% leaf_nodes,
  is_branch_direct = closest_vertex[colnames(cds)] %in% branch_nodes,
  monocle3_umap1 = reducedDims(cds)$UMAP[, 1],
  monocle3_umap2 = reducedDims(cds)$UMAP[, 2],
  stringsAsFactors = FALSE
)

write.csv(cell_out, paste0(output_prefix, "_cells.csv"), row.names = FALSE)

# 画轨迹图
png(paste0(output_prefix, "_trajectory_pseudotime.png"),
    width = 2200, height = 1800, res = 220)
print(plot_cells(cds, color_cells_by = "pseudotime",
                 label_leaves = TRUE, label_branch_points = TRUE,
                 graph_label_size = 1.5))
dev.off()

png(paste0(output_prefix, "_trajectory_partition.png"),
    width = 2200, height = 1800, res = 220)
print(plot_cells(cds, color_cells_by = "partition",
                 label_leaves = TRUE, label_branch_points = TRUE,
                 graph_label_size = 1.5))
dev.off()

cat("Monocle3 completed successfully.\n")
'''
    # 写 R 脚本
    with open(_r_script, "w", encoding="utf-8") as _f:
        _f.write(_r_code)
    print(f"R 脚本已生成: {_r_script}")

    # 调用 Rscript
    _mtx_path = os.path.join(MONOCLE3_WORK_DIR, "counts.mtx")
    _meta_path = os.path.join(MONOCLE3_WORK_DIR, "cell_meta.csv")
    _gene_path = os.path.join(MONOCLE3_WORK_DIR, "gene_anno.csv")
    _umap_path = os.path.join(MONOCLE3_WORK_DIR, "existing_umap.csv")
    _out_prefix = os.path.join(MONOCLE3_WORK_DIR, "monocle3")

    _cmd = [
        RSCRIPT_BIN, "--vanilla", _r_script,
        _mtx_path, _meta_path, _gene_path, _umap_path, _out_prefix,
        str(MONOCLE3_NUM_DIM), str(MONOCLE3_CORES),
        str(_top_cluster),
    ]
    print(f"正在运行: {' '.join(_cmd)}")

    _env = os.environ.copy()
    _env["R_PROFILE_USER"] = ""
    _env["R_ENVIRON_USER"] = ""

    try:
        _res = subprocess.run(
            _cmd, capture_output=True, text=True,
            env=_env, timeout=1800,  # Monocle3 可能需要较长时间
        )
        # 保存 stdout/stderr
        _stdout_path = os.path.join(MONOCLE3_WORK_DIR, "stdout.log")
        _stderr_path = os.path.join(MONOCLE3_WORK_DIR, "stderr.log")
        with open(_stdout_path, "w") as _f:
            _f.write(_res.stdout or "")
        with open(_stderr_path, "w") as _f:
            _f.write(_res.stderr or "")

        if _res.returncode != 0:
            print(f"Monocle3 运行失败 (exitcode={_res.returncode})")
            print(f"  STDOUT: {_stdout_path}")
            print(f"  STDERR: {_stderr_path}")
            print(f"  STDERR 尾部: {(_res.stderr or '')[-300:]}")
            _monocle3_success = False
        else:
            print("Monocle3 运行成功")
            _monocle3_success = True
    except subprocess.TimeoutExpired:
        print("Monocle3 运行超时（>30 分钟），已终止")
        _monocle3_success = False
else:
    print("Monocle3 不可用，跳过")
    _monocle3_success = False


In [ ]:
# 读取 Monocle3 结果，写回 adata.obs 和 adata.obsm。

if _monocle3_available and _monocle3_success:
    _cells_csv = os.path.join(MONOCLE3_WORK_DIR, "monocle3_cells.csv")
    if os.path.exists(_cells_csv):
        _m3 = pd.read_csv(_cells_csv).set_index("cell_id")
        # 对齐细胞索引
        _m3 = _m3.reindex(adata.obs_names)

        if "pseudotime" in _m3.columns:
            adata.obs["pseudotime_monocle3_v1"] = pd.to_numeric(
                _m3["pseudotime"], errors="coerce"
            ).values
            _pt = adata.obs["pseudotime_monocle3_v1"]
            print(f"Monocle3 pseudotime 已写入 adata.obs")
            print(f"  均值: {_pt.mean():.3f}  |  范围: [{_pt.min():.3f}, {_pt.max():.3f}]")
            print(f"  非 NA 细胞: {_pt.notna().sum()}/{len(_pt)}")

        if "monocle3_cluster" in _m3.columns:
            adata.obs["monocle3_cluster"] = _m3["monocle3_cluster"].astype(str).values

        if "monocle3_partition" in _m3.columns:
            adata.obs["monocle3_partition"] = _m3["monocle3_partition"].astype(str).values

        if "is_leaf_direct" in _m3.columns:
            adata.obs["monocle3_is_leaf"] = _m3["is_leaf_direct"].fillna(False).values

        # Monocle3 UMAP（可能与 Python 侧 UMAP 略有差异，保存为独立 key）
        if {"monocle3_umap1", "monocle3_umap2"}.issubset(_m3.columns):
            adata.obsm["X_monocle3_umap"] = _m3[[
                "monocle3_umap1", "monocle3_umap2"
            ]].to_numpy(dtype=np.float32)
            print("Monocle3 UMAP 已写入 adata.obsm['X_monocle3_umap']")

        # 复制 R 产出的 figure 到 results/figures/
        for _fname in [
            "monocle3_trajectory_pseudotime.png",
            "monocle3_trajectory_partition.png",
        ]:
            _src = os.path.join(MONOCLE3_WORK_DIR, _fname)
            if os.path.exists(_src):
                _dst = os.path.join("results", "figures", f"stage7_{_fname}")
                shutil.copy2(_src, _dst)
                print(f"Monocle3 figure 已复制: {_dst}")
    else:
        print(f"Monocle3 结果文件不存在: {_cells_csv}")
else:
    print("Monocle3 结果不可用，pseudotime_monocle3_v1 不会写入 adata.obs")


In [ ]:
# 内存自检 -- 确保 adata.X 稀疏性/精度在拟时序流程中未被破坏。
# CytoTRACE 和 Monocle3 可能创建额外 dense 矩阵，
# 但 adata.X 本身应始终保留为稀疏浮点格式。
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量被破坏: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 是 sparse CSR float32")

# 检查新增 obs 列
_new_cols = ["entropy", "cytotrace_score", "root_score"]
for _c in _new_cols:
    if _c in adata.obs.columns:
        _v = adata.obs[_c]
        print(f"  {_c}: mean={_v.mean():.4f}, non-NA={_v.notna().sum()}/{len(_v)}")
    else:
        print(f"  {_c}: 未写入")


In [ ]:
# 写出 checkpoint。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写出 {OUTPUT_PATH}")
assert os.path.exists(OUTPUT_PATH), f"输出不存在: {OUTPUT_PATH}"
print(f"已验证: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")


In [ ]:
# 释放内存。
del adata
gc.collect()
print("内存已释放")
